In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import re
import string


data_fake = pd.read_csv('Fake.csv')
data_true = pd.read_csv('True.csv')

data_fake["class"] = 0
data_true["class"] = 1

data_fake_manual_testing = data_fake.tail(10).copy()
data_fake = data_fake.iloc[:-10] 

data_true_manual_testing = data_true.tail(10).copy()
data_true = data_true.iloc[:-10] 

data_fake_manual_testing['class'] = 0
data_true_manual_testing['class'] = 1

data_merge = pd.concat([data_fake, data_true], axis = 0)
data = data_merge.drop(['title','subject', 'date'], axis = 1)

data = data.sample(frac = 1)
data.reset_index(inplace = True)
data.drop(['index'], axis = 1, inplace = True)

def wordopt(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)  
    text = re.sub(r"\W", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)  
    text = re.sub(r"\n", "", text)
    text = re.sub(r"\w*\d\w*", "", text)  
    return text

data['text'] = data['text'].apply(wordopt)

x = data['text']
y = data['class']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

from sklearn.feature_extraction.text import TfidfVectorizer
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)


from sklearn.linear_model import LogisticRegression
LR = LogisticRegression()
LR.fit(xv_train, y_train)
pred_lr = LR.predict(xv_test)
print("--- Logistic Regression ---")
print("Accuracy:", LR.score(xv_test, y_test))
print(classification_report(y_test, pred_lr))

from sklearn.tree import DecisionTreeClassifier
DT = DecisionTreeClassifier()
DT.fit(xv_train, y_train)
pred_dt = DT.predict(xv_test)
print("--- Decision Tree ---")
print("Accuracy:", DT.score(xv_test, y_test))
print(classification_report(y_test, pred_dt))

from sklearn.ensemble import GradientBoostingClassifier
GB = GradientBoostingClassifier(random_state = 0)
GB.fit(xv_train, y_train)
pred_gb = GB.predict(xv_test)
print("--- Gradient Boosting ---")
print("Accuracy:", GB.score(xv_test, y_test))
print(classification_report(y_test, pred_gb))

from sklearn.ensemble import RandomForestClassifier
RF = RandomForestClassifier(random_state = 0)
RF.fit(xv_train, y_train)
pred_rf = RF.predict(xv_test)
print("--- Random Forest ---")
print("Accuracy:", RF.score(xv_test, y_test))
print(classification_report(y_test, pred_rf))



def output_lable(n):
    if n == 0:
        return "Fake News"
    elif n == 1:
        return "Not A Fake News"

def manual_testing(news):
    testing_news = {"text":[news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt) 
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    
    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GBC = GB.predict(new_xv_test)  
    pred_RF = RF.predict(new_xv_test) 
    
    print("\n==================================")
    print("LR Prediction  : {}".format(output_lable(pred_LR[0])))
    print("DT Prediction  : {}".format(output_lable(pred_DT[0])))
    print("GBC Prediction : {}".format(output_lable(pred_GBC[0])))
    print("RF Prediction  : {}".format(output_lable(pred_RF[0])))
    print("==================================")



--- Logistic Regression ---
Accuracy: 0.9874331550802139
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      5801
           1       0.98      0.99      0.99      5419

    accuracy                           0.99     11220
   macro avg       0.99      0.99      0.99     11220
weighted avg       0.99      0.99      0.99     11220

--- Decision Tree ---
Accuracy: 0.996078431372549
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5801
           1       1.00      1.00      1.00      5419

    accuracy                           1.00     11220
   macro avg       1.00      1.00      1.00     11220
weighted avg       1.00      1.00      1.00     11220

--- Gradient Boosting ---
Accuracy: 0.9951871657754011
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      5801
           1       0.99      1.00      1.00      5419

    accuracy        

In [5]:
news = "BREAKING: Barack Obama was just arrested in his home early this morning for illegally wiretapping Donald Trump's phones during the election. Secret Service agents reportedly found multiple illegal recording devices hidden inside Trump Tower. The former president is currently being held in a federal facility without bail and is facing charges of treason."

manual_testing(news)


LR Prediction  : Fake News
DT Prediction  : Fake News
GBC Prediction : Fake News
RF Prediction  : Fake News


In [6]:
import pickle

# Vectorizer සහ Model එක ෆයිල් විදිහට save කිරීම
with open('vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorization, file)

with open('model.pkl', 'wb') as file:
    pickle.dump(LR, file)

print("Models successfully saved!")

Models successfully saved!
